### Humanoid

successor of Robot


In [4]:
import threading
from enum import Enum, auto
from time import sleep
from random import randint
from math import isclose, cos, sin, radians
from typing import Dict, List

In [ ]:
class Status(Enum):
    # belongs in the battery class but typing won't allow  self.Status as a type
    CHARGING = auto()
    NOT_CHARGING = auto()

    # battery health
    GOOD = auto()

    # humanoid Status will move them elsewhere
    ON = auto()

In [ ]:
# Battery


class Battery:
    def __init__(self, percentage: int = 0):
        self.percentage: float = percentage
        self.status: Status = Status.NOT_CHARGING
        self.charging_time: int = 60  # in seconds

        self.health: Status = (
            Status.GOOD
        )  # "good"|"bad" lol weather or not, to slow the charging (add inaccuracies to charging time)

    def _mod_battery(self, percentage: int = 1) -> bool:
        if (self.percentage + percentage) in range(0, 100 + 1):
            self.percentage = +percentage
            self.status = Status.NOT_CHARGING
            return True
        return False

    def charge(self, charge_length: int) -> bool:
        # charge_length: how long has it been charging or do you want to charge

        # really would like to instead have a method connect-charger that just runs in the bg and and ticks at time intervals updating percentage if still charging stops when t.cancel() is called or something closer to actual charging
        self.status = Status.CHARGING
        timer = threading.Timer(
            interval=charge_length,
            function=self._mod_battery,
            kwargs={"percentage": (charge_length / self.charging_time) * 100},
        )
        # guess a better way would be to say how long a full charge takes then define a function to map the appropriate percentage based on the interval the length of our current timer (lol bette)
        timer.start()
        return True

    def use_up(self, cost: float):
        # based on how it goes was thinking of using time to use up battery as the action is being performed at the rate of cost
        if (self.percentage - cost) >= 0:
            self.percentage -= cost
            return True
        return False

    def __str__(self):
        return f"{self.status} :{self.percentage}"

In [ ]:
# test : battery charging and use

me = Battery()
print(me.status, " ; ", me.percentage)  # initial

me.charge(3)
print(me.status, " ; ", me.percentage)  # action

sleep(4)
print(me.status, " ; ", me.percentage)  # results

me.use_up(5)
print(me.status, " ; ", me.percentage)  # results 1

me.use_up(5)

In [ ]:
# Humanoid

class Direction:
    def __init__(self, angle = 0):
        self.angle = angle % 360

    def _turn(self, right:bool=True):
        self.angle = (self.angle + (1 if right else -1)*90) % 360

    def right(self):
        self._turn()

    def left(self):
        self._turn(False)




class Body:
    def __init__(self):
        self.angle = Direction()
        self.head:Coord = Coord(0, 0)
        self.tail:Coord = self.head  # f it start as a point, then stretch, snake

    def move(self):
        # ask your world to move in a direction 1 unit, but idk world so relative to self 
        new = self.head
        new.x =+ cos(radians(self.angle))
        new.y =+ sin(radians(self.angle))


class Humanoid:
    def __init__(self):
        self.battery: Battery = Battery()
        self.status: Status = Status.ON
        self.body: List[Coord] = Body()

    def charge_battery(self, charge_length: int):
        self.status = Status.CHARGING
        result = self.battery.charge(charge_length)
        return result

    def __str__(self):
        return f"{self.status} : {self.battery})"

    def __repr__(self):
        return f"Humanoid({self.battery})"

    

In [ ]:
# test : humanoid

me = Humanoid()

print(me.battery.percentage, me.status, sep=" : ")  # initial

me.charge_battery(3)
print(me.battery.percentage, me.status, sep=" : ")  # action

sleep(4)
print(me.battery.percentage, me.status, sep=" : ")  # results

In [ ]:
# world


class Coord:
    def __init__(self, x: float, y: float):
        self.x = x
        self.y = y

    def __eq__(self, other: object) -> bool:
        """Check equality of two coordinates."""
        if not isinstance(other, Coord):
            return NotImplemented
        return isclose(self.x, other.x, abs_tol=1e-9) and isclose(
            self.y, other.y, abs_tol=1e-9
        )

    def __hash__(self) -> int:
        """Return a hash of the angle for hashable collections."""
        return hash((self.x, self.y))

    def __repr__(self):
        return f"Coord(x={self.x}, y={self.y})"


class World:
    def __init__(self):
        # valid coords x : -50 -> 50 ; y : -50 -> 50
        self.x_range = range(-5, 5+1)
        self.y_range = range(-5, 5+1)

        self.humanoids: Dict[Coord, Humanoid] = {}

    def spawn_humanoid(self):
        coord = Coord(randint(self.x_range.start, self.x_range.stop-1), y=randint(self.y_range.start, self.y_range.stop-1))

        if self.humanoids.get(coord) is None:
            self.humanoids[coord] = Humanoid()
            return True
        return False

In [ ]:
# test : world spawn

w = World()

print(w.humanoids)  # initial

print(w.spawn_humanoid())  # action
print(w.humanoids)
